# Peek at read-based taxonomy ↔ biosamples

Worked-example queries linking read-based taxonomy results
(Kraken2, GOTTCHA2, Centrifuge) to biosamples via
`nmdc_metadata.biosample_to_workflow_run`.

**Both directions:**
- §1–3: biosample → what taxa were detected (by method)
- §4–5: taxon name → which biosamples detected it (by method)

**Centrifuge** data is not yet ingested. §3 documents the expected
schema and query pattern for when it is.

In [1]:
spark = get_spark_session(app_name="peek_read_taxonomy_links", tenant_name="nmdc")
print(f"Spark version: {spark.version}")

Spark version: 4.0.1


### Preflight: check available read taxonomy tables

In [2]:
existing = {r["tableName"] for r in spark.sql("SHOW TABLES IN nmdc_results").toPandas().to_dict("records")}

checks = {
    'kraken2':   'kraken2_classification_report',
    'gottcha2':  'gottcha2_classification_report',
    'centrifuge': 'centrifuge_output_report_file',
}

available = {}
for key, tbl in checks.items():
    found = tbl in existing
    available[key] = found
    status = "OK" if found else "MISSING"
    print(f"{status}: nmdc_results.{tbl}")

OK: nmdc_results.kraken2_classification_report
OK: nmdc_results.gottcha2_classification_report
OK: nmdc_results.centrifuge_output_report_file


### Pick an anchor biosample

Find a biosample that has at least one Kraken2 result via the workflow bridge.

In [3]:
_anchor_df = spark.sql("""
    SELECT b2wr.biosample_id, b2wr.workflow_run_id, b2wr.workflow_type
    FROM   nmdc_results.kraken2_classification_report k
    JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
             ON b2wr.workflow_run_id = k.workflow_run_id
    LIMIT 1
""").toPandas()
if _anchor_df.empty:
    raise RuntimeError("no kraken2_classification_report row joins to biosample_to_workflow_run — verify both tables are populated")
anchor = _anchor_df.iloc[0]
BIOSAMPLE_ID  = anchor["biosample_id"]
KRAKEN2_RUN_ID = anchor["workflow_run_id"]
print(anchor.to_string())

biosample_id                 nmdc:bsm-15-nr7g3b70
workflow_run_id          nmdc:wfrbt-11-2b5pk711.1
workflow_type      nmdc:ReadBasedTaxonomyAnalysis


### 1. Biosample → Kraken2 taxa

Species-level hits (rank = 'S') with at least 1 direct read, ordered by % abundance.
Kraken2 has 36M rows total — always filter by workflow_run_id.

In [4]:
spark.sql(f"""
    SELECT rank, name, taxid, pct_clade, clade_reads, direct_reads
    FROM   nmdc_results.kraken2_classification_report
    WHERE  workflow_run_id = '{KRAKEN2_RUN_ID}'
      AND  rank = 'S'
      AND  direct_reads > 0
    ORDER BY pct_clade DESC
    LIMIT 20
""").toPandas()

,rank,name,taxid,pct_clade,clade_reads,direct_reads
0,S,Bradyrhizobium erythrophlei,1437360,0.96,68761,68761
1,S,Bradyrhizobium lablabi,722472,0.30,21146,21146
2,S,Rhodopseudomonas palustris,1076,0.17,12382,3456
3,S,...,9606,0.13,9396,9396
4,S,Bradyrhizobium icense,1274631,0.10,7024,7024
5,S,Afipia sp. GAS231,1882747,0.09,6736,6736
6,S,Bradyrhizobium sp. SK17,2057741,0.08,5371,5371
7,S,Bradyrhizobium diazoefficiens,1355477,0.06,4042,3908
8,S,Rhodoplanes sp. Z2-YC6860,674703,0.06,3976,3976
9,S,Bradyrhizobium ottawaense,931866,0.04,2901,2901


### 2. Biosample → GOTTCHA2 taxa

Species-level hits ordered by relative abundance. GOTTCHA2 uses a unique linear-length
scoring approach that reduces false positives vs. Kraken2.

In [5]:
if not available['gottcha2']:
    print("SKIP: gottcha2_classification_report not available")
else:
    g2_run = spark.sql(f"""
        SELECT b2wr.workflow_run_id
        FROM   nmdc_metadata.biosample_to_workflow_run b2wr
        JOIN   nmdc_results.gottcha2_classification_report g
                 ON g.workflow_run_id = b2wr.workflow_run_id
        WHERE  b2wr.biosample_id = '{BIOSAMPLE_ID}'
        LIMIT 1
    """).toPandas()
    if g2_run.empty:
        print(f"No GOTTCHA2 results for {BIOSAMPLE_ID} — try a different biosample")
        GOTTCHA2_RUN_ID = None
    else:
        GOTTCHA2_RUN_ID = g2_run.iloc[0]["workflow_run_id"]
        spark.sql(f"""
            SELECT LEVEL, NAME, TAXID, READ_COUNT, REL_ABUNDANCE
            FROM   nmdc_results.gottcha2_classification_report
            WHERE  workflow_run_id = '{GOTTCHA2_RUN_ID}'
              AND  LEVEL = 'species'
            ORDER BY REL_ABUNDANCE DESC
            LIMIT 20
        """).toPandas()

### 3. Biosample → Centrifuge taxa

`nmdc_results.centrifuge_output_report_file` — 20,564,570 rows, one per (taxon, workflow run).

| Column | Type | Description |
|---|---|---|
| `name` | string | Taxon name |
| `taxID` | int | NCBI taxon ID |
| `taxRank` | string | Taxonomic rank |
| `genomeSize` | int | Reference genome size |
| `numReads` | int | Reads classified to this taxon |
| `numUniqueReads` | int | Reads uniquely classified here |
| `abundance` | float | Fraction of total reads |
| `workflow_run_id` | string | Join key to `biosample_to_workflow_run` |

In [6]:
if not available['centrifuge']:
    print("centrifuge_output_report_file: NOT YET INGESTED")
    print("Run fetch_taxonomy_summaries.ipynb + ingest_taxonomy_summaries.ipynb to populate.")
else:
    centrifuge_run = spark.sql(f"""
        SELECT b2wr.workflow_run_id
        FROM   nmdc_metadata.biosample_to_workflow_run b2wr
        JOIN   nmdc_results.centrifuge_output_report_file c
                 ON c.workflow_run_id = b2wr.workflow_run_id
        WHERE  b2wr.biosample_id = '{BIOSAMPLE_ID}'
        LIMIT 1
    """).toPandas()
    if centrifuge_run.empty:
        print(f"No Centrifuge results for {BIOSAMPLE_ID}")
    else:
        run_id = centrifuge_run.iloc[0]["workflow_run_id"]
        spark.sql(f"""
            SELECT name, taxID, taxRank, numReads, numUniqueReads, abundance
            FROM   nmdc_results.centrifuge_output_report_file
            WHERE  workflow_run_id = '{run_id}'
              AND  abundance > 0
            ORDER BY abundance DESC
            LIMIT 20
        """).toPandas()

### 4. Taxon → biosamples (Kraken2)

Which biosamples had a given taxon detected? Filter by name and a minimum clade percentage.
Aggregates across all Kraken2 runs for each biosample.

In [7]:
TARGET_TAXON = "Bacteroides"  # substitute any taxon name

if not available['kraken2']:
    print("SKIP: kraken2_classification_report not available")
else:
    spark.sql(f"""
        SELECT b2wr.biosample_id,
               bsm.env_broad_scale_term_id,
               bsm.geo_loc_name_has_raw_value,
               MAX(k.pct_clade)   AS max_pct_clade,
               SUM(k.clade_reads) AS total_clade_reads
        FROM   nmdc_results.kraken2_classification_report k
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = k.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  TRIM(k.name) = '{TARGET_TAXON}'
          AND  k.pct_clade > 0.1
        GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
        ORDER BY max_pct_clade DESC
        LIMIT 20
    """).toPandas()

### 5. Taxon → biosamples (GOTTCHA2)

In [8]:
if not available['gottcha2']:
    print("SKIP: gottcha2_classification_report not available")
else:
    spark.sql(f"""
        SELECT b2wr.biosample_id,
               bsm.env_broad_scale_term_id,
               bsm.geo_loc_name_has_raw_value,
               MAX(g.REL_ABUNDANCE)  AS max_rel_abundance,
               SUM(g.READ_COUNT)     AS total_read_count
        FROM   nmdc_results.gottcha2_classification_report g
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = g.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  TRIM(g.NAME) = '{TARGET_TAXON}'
          AND  g.REL_ABUNDANCE > 0
        GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
        ORDER BY max_rel_abundance DESC
        LIMIT 20
    """).toPandas()

### 6. Taxon → biosamples (Centrifuge)

Which biosamples had a given taxon detected by Centrifuge?
Aggregates across all Centrifuge runs for each biosample.

In [9]:
if not available['centrifuge']:
    print("SKIP: centrifuge_output_report_file not available")
else:
    spark.sql(f"""
        SELECT b2wr.biosample_id,
               bsm.env_broad_scale_term_id,
               bsm.geo_loc_name_has_raw_value,
               MAX(c.abundance)        AS max_abundance,
               SUM(c.numReads)         AS total_reads
        FROM   nmdc_results.centrifuge_output_report_file c
        JOIN   nmdc_metadata.biosample_to_workflow_run b2wr
                 ON b2wr.workflow_run_id = c.workflow_run_id
        JOIN   nmdc_metadata.biosample_set bsm
                 ON bsm.id = b2wr.biosample_id
        WHERE  TRIM(c.name) = '{TARGET_TAXON}'
          AND  c.abundance > 0
        GROUP BY b2wr.biosample_id, bsm.env_broad_scale_term_id, bsm.geo_loc_name_has_raw_value
        ORDER BY max_abundance DESC
        LIMIT 20
    """).toPandas()